# Semicon Restoration — Final Clean Notebook
This notebook is a clean, single-pass walkthrough: mount Drive, sanity-check the data, define the model, load (or train) the checkpoint, generate the standalone `evaluate.py`/`train.py`/`requirements.txt` files, and produce the final restored test outputs.

**Only thing you need to change:** `BASE_DIR` in the next-but-one cell, to point at your friend's Drive folder structure (must contain `train/GT`, `train/NoisyLR`, `dataset_split.csv`, and — for real-test inference — a `Test_NoisyLR` folder with a `NoisyLR` subfolder).

Run top to bottom. The training cell is skippable if a checkpoint already exists — a check cell tells you which.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# >>> CHANGE THIS to match your friend's Drive structure <<<
BASE_DIR = "/content/drive/MyDrive/Semicon"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
GT_DIR = os.path.join(TRAIN_DIR, "GT")
NOISYLR_DIR = os.path.join(TRAIN_DIR, "NoisyLR")
SPLIT_PATH = os.path.join(BASE_DIR, "dataset_split.csv")
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "model_v2_best.pth")
REAL_TEST_DIR = os.path.join(BASE_DIR, "Test_NoisyLR", "NoisyLR")  # adjust if your test folder is named differently
OUTPUT_DIR = os.path.join(BASE_DIR, "final_submission_outputs")

for p in [GT_DIR, NOISYLR_DIR, SPLIT_PATH]:
    print(p, "->", "EXISTS" if os.path.exists(p) else "MISSING")

In [ ]:
!pip install -q lpips scikit-image

## Dataset

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

NOISY_MEAN = 0.433536
NOISY_STD = 0.284787

class SemiconductorDataset(Dataset):
    def __init__(self, split, split_csv, gt_dir, noisy_dir):
        self.gt_dir = gt_dir
        self.noisy_dir = noisy_dir
        df = pd.read_csv(split_csv)
        self.df = df[df["split"] == split].reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        filename = self.df.loc[idx, "filename"]
        lr = np.load(os.path.join(self.noisy_dir, filename)).astype(np.float32)
        gt = np.load(os.path.join(self.gt_dir, filename)).astype(np.float32)
        lr = (lr - NOISY_MEAN) / NOISY_STD
        return torch.from_numpy(lr).unsqueeze(0), torch.from_numpy(gt).unsqueeze(0)

train_dataset = SemiconductorDataset("train", SPLIT_PATH, GT_DIR, NOISYLR_DIR)
val_dataset = SemiconductorDataset("validation", SPLIT_PATH, GT_DIR, NOISYLR_DIR)
test_dataset = SemiconductorDataset("test", SPLIT_PATH, GT_DIR, NOISYLR_DIR)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print("Train:", len(train_dataset), "Val:", len(val_dataset), "Test:", len(test_dataset))

## Model

In [ ]:
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
        )
    def forward(self, x):
        return x + self.block(x)

class RestorationNetV1(nn.Module):
    def __init__(self, num_features=64, num_blocks=8):
        super().__init__()
        self.head = nn.Conv2d(1, num_features, kernel_size=3, padding=1)
        self.body = nn.Sequential(*[ResidualBlock(num_features) for _ in range(num_blocks)])
        self.body_conv = nn.Conv2d(num_features, num_features, kernel_size=3, padding=1)
        self.upsample = nn.Sequential(
            nn.Conv2d(num_features, num_features * 4, kernel_size=3, padding=1),
            nn.PixelShuffle(2),
            nn.ReLU(inplace=True),
        )
        self.tail = nn.Conv2d(num_features, 1, kernel_size=3, padding=1)

    def forward(self, x):
        features = self.head(x)
        body = self.body(features)
        body = self.body_conv(body)
        features = features + body
        features = self.upsample(features)
        output = self.tail(features)
        return torch.sigmoid(output)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RestorationNetV1(num_features=64, num_blocks=8).to(device)
print(model)
print("Device:", device)

## Train (skip if checkpoint already exists)
This trains for 30 epochs (~30-90 min on a T4) using the combined L1+SSIM loss. **If the check below prints `True`, skip the training cell and go straight to the checkpoint-loading cell.**

In [ ]:
ckpt_exists = os.path.exists(CHECKPOINT_PATH)
print("Checkpoint already exists:", ckpt_exists)
print("-> If True: skip the training cell, run the checkpoint-loading cell instead.")
print("-> If False: run the training cell (real training run).")

In [ ]:
# TRAINING CELL — skip this if ckpt_exists was True above
import torch.nn.functional as F
import math, time

def gaussian_window(window_size, sigma, channels):
    coords = torch.arange(window_size).float() - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    window_1d = g.unsqueeze(1)
    window_2d = window_1d @ window_1d.t()
    return window_2d.expand(channels, 1, window_size, window_size).contiguous()

def ssim_loss(prediction, target, window_size=11, sigma=1.5):
    prediction = torch.clamp(prediction, 0.0, 1.0)
    target = torch.clamp(target, 0.0, 1.0)
    channels = prediction.size(1)
    window = gaussian_window(window_size, sigma, channels).to(prediction.device)
    mu_pred = F.conv2d(prediction, window, padding=window_size // 2, groups=channels)
    mu_target = F.conv2d(target, window, padding=window_size // 2, groups=channels)
    mu_pred_sq, mu_target_sq, mu_pred_target = mu_pred.pow(2), mu_target.pow(2), mu_pred * mu_target
    sigma_pred_sq = F.conv2d(prediction * prediction, window, padding=window_size // 2, groups=channels) - mu_pred_sq
    sigma_target_sq = F.conv2d(target * target, window, padding=window_size // 2, groups=channels) - mu_target_sq
    sigma_pred_target = F.conv2d(prediction * target, window, padding=window_size // 2, groups=channels) - mu_pred_target
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    ssim_map = ((2 * mu_pred_target + C1) * (2 * sigma_pred_target + C2)) / \
               ((mu_pred_sq + mu_target_sq + C1) * (sigma_pred_sq + sigma_target_sq + C2))
    return 1.0 - ssim_map.mean()

def restoration_loss_v2(pred, target, alpha=0.85, beta=0.15):
    l1 = F.l1_loss(pred, target)
    ssim = ssim_loss(pred, target)
    return alpha * l1 + beta * ssim, l1, ssim

def calculate_psnr(pred, target):
    pred, target = torch.clamp(pred, 0., 1.), torch.clamp(target, 0., 1.)
    mse = torch.mean((pred - target) ** 2)
    return float("inf") if mse == 0 else 10 * math.log10(1.0 / mse.item())

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

best_psnr = -float("inf")
NUM_EPOCHS = 30

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()
    model.train()
    tr_loss = tr_psnr = 0.0
    for lr_b, gt_b in train_loader:
        lr_b, gt_b = lr_b.to(device), gt_b.to(device)
        pred = model(lr_b)
        loss, _, _ = restoration_loss_v2(pred, gt_b)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tr_loss += loss.item(); tr_psnr += calculate_psnr(pred, gt_b)
    tr_loss /= len(train_loader); tr_psnr /= len(train_loader)

    model.eval()
    val_loss = val_psnr = 0.0
    with torch.no_grad():
        for lr_b, gt_b in val_loader:
            lr_b, gt_b = lr_b.to(device), gt_b.to(device)
            pred = model(lr_b)
            loss, _, _ = restoration_loss_v2(pred, gt_b)
            val_loss += loss.item(); val_psnr += calculate_psnr(pred, gt_b)
    val_loss /= len(val_loader); val_psnr /= len(val_loader)
    scheduler.step(val_psnr)

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | train_loss {tr_loss:.4f} train_psnr {tr_psnr:.2f} | "
          f"val_loss {val_loss:.4f} val_psnr {val_psnr:.2f} | {time.time()-start:.1f}s")

    if val_psnr > best_psnr:
        best_psnr = val_psnr
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(), "val_psnr": val_psnr}, CHECKPOINT_PATH)
        print(f"  -> new best (val_psnr {val_psnr:.4f}), saved.")

print("\nBest val PSNR:", best_psnr)

In [ ]:
# CHECKPOINT-LOADING CELL — run this instead of training if ckpt_exists was True
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Loaded checkpoint. Val PSNR at save time:", checkpoint.get("val_psnr"))

## Evaluate on held-out test split (PSNR / SSIM / LPIPS)

In [ ]:
from skimage.metrics import structural_similarity
import lpips

lpips_model = lpips.LPIPS(net="alex").to(device)
lpips_model.eval()

@torch.no_grad()
def evaluate_test_set(model, loader, device):
    model.eval()
    psnr_scores, ssim_scores, lpips_scores = [], [], []
    for lr_b, gt_b in loader:
        lr_b, gt_b = lr_b.to(device), gt_b.to(device)
        pred = torch.clamp(model(lr_b), 0.0, 1.0)
        for i in range(pred.shape[0]):
            p = pred[i, 0].cpu().numpy()
            g = gt_b[i, 0].cpu().numpy()
            psnr_scores.append(10 * np.log10(1.0 / np.mean((p - g) ** 2)))
            ssim_scores.append(structural_similarity(g, p, data_range=1.0))
        pred_lp = (pred.repeat(1, 3, 1, 1) * 2 - 1)
        gt_lp = (gt_b.repeat(1, 3, 1, 1) * 2 - 1)
        lpips_scores.append(lpips_model(pred_lp, gt_lp).mean().item())
    return np.mean(psnr_scores), np.mean(ssim_scores), np.mean(lpips_scores)

test_psnr, test_ssim, test_lpips = evaluate_test_set(model, test_loader, device)
print(f"Test PSNR : {test_psnr:.4f} dB")
print(f"Test SSIM : {test_ssim:.6f}")
print(f"Test LPIPS: {test_lpips:.6f}")

## Generate restored outputs on the real KLA test set

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.eval()

test_files = sorted(f for f in os.listdir(REAL_TEST_DIR) if f.endswith(".npy"))
print(f"Found {len(test_files)} real test files.")

with torch.no_grad():
    for i, filename in enumerate(test_files):
        lr = np.load(os.path.join(REAL_TEST_DIR, filename)).astype(np.float32)
        lr_norm = (lr - NOISY_MEAN) / NOISY_STD
        lr_tensor = torch.from_numpy(lr_norm).unsqueeze(0).unsqueeze(0).to(device)
        pred = torch.clamp(model(lr_tensor), 0.0, 1.0)
        np.save(os.path.join(OUTPUT_DIR, filename), pred[0, 0].cpu().numpy())
        if (i + 1) % 50 == 0:
            print(f"Processed {i+1}/{len(test_files)}")

print("Done. Outputs saved to:", OUTPUT_DIR)

## Write the mandatory repo files
This writes `evaluate.py` (the standalone benchmarking script KLA will run as-is), `train.py`, and `requirements.txt` directly to your Drive `BASE_DIR`, ready to be committed to the GitHub repo alongside the checkpoint and outputs folder.

In [ ]:
evaluate_py = r"""\"\"\"
evaluate.py — KLA benchmarking script.

Loads the trained RestorationNetV2 checkpoint and runs inference on every
.npy file in --test_dir, writing the restored (2x super-resolved,
denoised) output for each to --output_dir under the same filename.

This script is meant to be run AS-IS, with no manual edits, other than
supplying --test_dir and --output_dir:

    python evaluate.py --test_dir /path/to/test_images --output_dir /path/to/save/outputs

By default it loads the checkpoint from ./checkpoints/model_v2_best.pth
(relative to this script's own location, so it works no matter where the
repo is cloned to). Override with --checkpoint if needed.
\"\"\"

import os
import argparse
import time

import numpy as np
import torch
import torch.nn as nn

# ============================================================
# Normalization constants (must match training preprocessing)
# ============================================================
NOISY_MEAN = 0.433536
NOISY_STD = 0.284787


# ============================================================
# Model — RestorationNetV2 (same architecture as V1, trained
# with the combined L1 + SSIM loss). Keys match the saved
# checkpoint's state_dict exactly.
# ============================================================
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
        )

    def forward(self, x):
        return x + self.block(x)


class RestorationNetV1(nn.Module):
    \"\"\"Name kept as RestorationNetV1 to match the trained checkpoint's
    module names — this is the architecture actually used for V2 training.\"\"\"

    def __init__(self, num_features=64, num_blocks=8):
        super().__init__()

        self.head = nn.Conv2d(1, num_features, kernel_size=3, padding=1)

        self.body = nn.Sequential(
            *[ResidualBlock(num_features) for _ in range(num_blocks)]
        )

        self.body_conv = nn.Conv2d(num_features, num_features, kernel_size=3, padding=1)

        self.upsample = nn.Sequential(
            nn.Conv2d(num_features, num_features * 4, kernel_size=3, padding=1),
            nn.PixelShuffle(2),
            nn.ReLU(inplace=True),
        )

        self.tail = nn.Conv2d(num_features, 1, kernel_size=3, padding=1)

    def forward(self, x):
        features = self.head(x)
        body = self.body(features)
        body = self.body_conv(body)
        features = features + body
        features = self.upsample(features)
        output = self.tail(features)
        return torch.sigmoid(output)


# ============================================================
# Helpers
# ============================================================
def get_default_checkpoint():
    script_dir = os.path.dirname(os.path.abspath(__file__))
    return os.path.join(script_dir, "checkpoints", "model_v2_best.pth")


def parse_args():
    parser = argparse.ArgumentParser(description="Run inference with the trained restoration model")
    parser.add_argument("--test_dir", type=str, required=True,
                         help="Directory containing input NoisyLR .npy images")
    parser.add_argument("--output_dir", type=str, required=True,
                         help="Directory to write restored .npy outputs to")
    parser.add_argument("--checkpoint", type=str, default=None,
                         help="Path to model_v2_best.pth (default: ./checkpoints/model_v2_best.pth next to this script)")
    args = parser.parse_args()
    if args.checkpoint is None:
        args.checkpoint = get_default_checkpoint()
    return args


def load_model(checkpoint_path, device):
    print("=" * 60)
    print("LOADING MODEL")
    print("=" * 60)

    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

    model = RestorationNetV1().to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
        print("Checkpoint epoch:", checkpoint.get("epoch", "unknown"))
        if "val_psnr" in checkpoint:
            print(f"Validation PSNR at save time: {checkpoint['val_psnr']:.3f} dB")
    else:
        model.load_state_dict(checkpoint)

    model.eval()
    print("Model loaded successfully.")
    return model


@torch.no_grad()
def run_inference(model, test_dir, output_dir, device):
    os.makedirs(output_dir, exist_ok=True)

    test_files = sorted(f for f in os.listdir(test_dir) if f.endswith(".npy"))
    if len(test_files) == 0:
        raise RuntimeError(f"No .npy files found in {test_dir}")

    print()
    print("=" * 60)
    print("RUNNING INFERENCE")
    print("=" * 60)
    print("Input dir :", test_dir)
    print("Output dir:", output_dir)
    print("Images    :", len(test_files))
    print("Device    :", device)
    print()

    start = time.time()

    for i, filename in enumerate(test_files):
        lr = np.load(os.path.join(test_dir, filename)).astype(np.float32)

        # Same normalization used during training
        lr_norm = (lr - NOISY_MEAN) / NOISY_STD
        lr_tensor = torch.from_numpy(lr_norm).unsqueeze(0).unsqueeze(0).to(device)

        pred = model(lr_tensor)
        pred = torch.clamp(pred, 0.0, 1.0)
        pred_np = pred[0, 0].cpu().numpy().astype(np.float32)

        np.save(os.path.join(output_dir, filename), pred_np)

        if (i + 1) % 50 == 0 or (i + 1) == len(test_files):
            print(f"Processed {i + 1}/{len(test_files)}")

    elapsed = time.time() - start
    print()
    print(f"Done. {len(test_files)} images restored in {elapsed:.2f}s "
          f"({elapsed / len(test_files) * 1000:.2f} ms/image).")
    print(f"Outputs saved to: {output_dir}")


def main():
    args = parse_args()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = load_model(args.checkpoint, device)
    run_inference(model, args.test_dir, args.output_dir, device)


if __name__ == "__main__":
    main()
"""
with open(os.path.join(BASE_DIR, "evaluate.py"), "w") as f:
    f.write(evaluate_py)
print("Wrote", os.path.join(BASE_DIR, "evaluate.py"))

In [ ]:
train_py = r"""\"\"\"
train.py — Reproduces the full RestorationNetV2 training run from scratch.

Trains a residual CNN to jointly denoise and 2x super-resolve single-channel
semiconductor (wafer/SEM) images, using a combined L1 + SSIM loss.

Usage:
    python train.py \
        --gt_dir /path/to/train/GT \
        --noisy_dir /path/to/train/NoisyLR \
        --split_csv /path/to/dataset_split.csv \
        --checkpoint_dir /path/to/checkpoints \
        --epochs 30
\"\"\"

import os
import time
import math
import argparse

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

NOISY_MEAN = 0.433536
NOISY_STD = 0.284787


# ============================================================
# Dataset
# ============================================================
class SemiconductorDataset(Dataset):
    def __init__(self, split, split_csv, gt_dir, noisy_dir):
        self.gt_dir = gt_dir
        self.noisy_dir = noisy_dir

        df = pd.read_csv(split_csv)
        self.df = df[df["split"] == split].reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        filename = self.df.loc[idx, "filename"]

        lr = np.load(os.path.join(self.noisy_dir, filename)).astype(np.float32)
        gt = np.load(os.path.join(self.gt_dir, filename)).astype(np.float32)

        lr = (lr - NOISY_MEAN) / NOISY_STD

        lr_tensor = torch.from_numpy(lr).unsqueeze(0)
        gt_tensor = torch.from_numpy(gt).unsqueeze(0)

        return lr_tensor, gt_tensor


# ============================================================
# Model
# ============================================================
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
        )

    def forward(self, x):
        return x + self.block(x)


class RestorationNetV1(nn.Module):
    def __init__(self, num_features=64, num_blocks=8):
        super().__init__()
        self.head = nn.Conv2d(1, num_features, kernel_size=3, padding=1)
        self.body = nn.Sequential(*[ResidualBlock(num_features) for _ in range(num_blocks)])
        self.body_conv = nn.Conv2d(num_features, num_features, kernel_size=3, padding=1)
        self.upsample = nn.Sequential(
            nn.Conv2d(num_features, num_features * 4, kernel_size=3, padding=1),
            nn.PixelShuffle(2),
            nn.ReLU(inplace=True),
        )
        self.tail = nn.Conv2d(num_features, 1, kernel_size=3, padding=1)

    def forward(self, x):
        features = self.head(x)
        body = self.body(features)
        body = self.body_conv(body)
        features = features + body
        features = self.upsample(features)
        output = self.tail(features)
        return torch.sigmoid(output)


# ============================================================
# Loss: combined L1 + SSIM
# ============================================================
def gaussian_window(window_size, sigma, channels):
    coords = torch.arange(window_size).float() - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    window_1d = g.unsqueeze(1)
    window_2d = window_1d @ window_1d.t()
    window = window_2d.expand(channels, 1, window_size, window_size).contiguous()
    return window


def ssim_loss(prediction, target, window_size=11, sigma=1.5):
    prediction = torch.clamp(prediction, 0.0, 1.0)
    target = torch.clamp(target, 0.0, 1.0)

    channels = prediction.size(1)
    window = gaussian_window(window_size, sigma, channels).to(prediction.device)

    mu_pred = F.conv2d(prediction, window, padding=window_size // 2, groups=channels)
    mu_target = F.conv2d(target, window, padding=window_size // 2, groups=channels)

    mu_pred_sq = mu_pred.pow(2)
    mu_target_sq = mu_target.pow(2)
    mu_pred_target = mu_pred * mu_target

    sigma_pred_sq = F.conv2d(prediction * prediction, window, padding=window_size // 2, groups=channels) - mu_pred_sq
    sigma_target_sq = F.conv2d(target * target, window, padding=window_size // 2, groups=channels) - mu_target_sq
    sigma_pred_target = F.conv2d(prediction * target, window, padding=window_size // 2, groups=channels) - mu_pred_target

    C1, C2 = 0.01 ** 2, 0.03 ** 2
    ssim_map = ((2 * mu_pred_target + C1) * (2 * sigma_pred_target + C2)) / \
               ((mu_pred_sq + mu_target_sq + C1) * (sigma_pred_sq + sigma_target_sq + C2))

    return 1.0 - ssim_map.mean()


def restoration_loss_v2(prediction, target, alpha=0.85, beta=0.15):
    l1 = F.l1_loss(prediction, target)
    ssim = ssim_loss(prediction, target)
    total = alpha * l1 + beta * ssim
    return total, l1, ssim


def calculate_psnr(pred, target):
    pred = torch.clamp(pred, 0.0, 1.0)
    target = torch.clamp(target, 0.0, 1.0)
    mse = torch.mean((pred - target) ** 2)
    if mse == 0:
        return float("inf")
    return 10 * math.log10(1.0 / mse.item())


# ============================================================
# Train / validate loops
# ============================================================
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, total_psnr = 0.0, 0.0

    for lr, gt in loader:
        lr, gt = lr.to(device, non_blocking=True), gt.to(device, non_blocking=True)

        pred = model(lr)
        loss, _, _ = restoration_loss_v2(pred, gt)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_psnr += calculate_psnr(pred, gt)

    n = len(loader)
    return total_loss / n, total_psnr / n


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    total_loss, total_psnr = 0.0, 0.0

    for lr, gt in loader:
        lr, gt = lr.to(device, non_blocking=True), gt.to(device, non_blocking=True)
        pred = model(lr)
        loss, _, _ = restoration_loss_v2(pred, gt)
        total_loss += loss.item()
        total_psnr += calculate_psnr(pred, gt)

    n = len(loader)
    return total_loss / n, total_psnr / n


# ============================================================
# Main
# ============================================================
def parse_args():
    parser = argparse.ArgumentParser(description="Train RestorationNetV2 from scratch")
    parser.add_argument("--gt_dir", type=str, required=True)
    parser.add_argument("--noisy_dir", type=str, required=True)
    parser.add_argument("--split_csv", type=str, required=True)
    parser.add_argument("--checkpoint_dir", type=str, required=True)
    parser.add_argument("--epochs", type=int, default=30)
    parser.add_argument("--batch_size", type=int, default=16)
    parser.add_argument("--lr", type=float, default=2e-4)
    return parser.parse_args()


def main():
    args = parse_args()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    os.makedirs(args.checkpoint_dir, exist_ok=True)

    train_dataset = SemiconductorDataset("train", args.split_csv, args.gt_dir, args.noisy_dir)
    val_dataset = SemiconductorDataset("validation", args.split_csv, args.gt_dir, args.noisy_dir)

    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True,
                               num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False,
                             num_workers=2, pin_memory=True)

    model = RestorationNetV1(num_features=64, num_blocks=8).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

    best_psnr = -float("inf")
    ckpt_path = os.path.join(args.checkpoint_dir, "model_v2_best.pth")

    for epoch in range(1, args.epochs + 1):
        start = time.time()

        train_loss, train_psnr = train_one_epoch(model, train_loader, optimizer, device)
        val_loss, val_psnr = validate(model, val_loader, device)
        scheduler.step(val_psnr)

        elapsed = time.time() - start
        print(f"Epoch {epoch:02d}/{args.epochs} | "
              f"train_loss {train_loss:.4f} train_psnr {train_psnr:.2f} | "
              f"val_loss {val_loss:.4f} val_psnr {val_psnr:.2f} | "
              f"{elapsed:.1f}s")

        if val_psnr > best_psnr:
            best_psnr = val_psnr
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "val_psnr": val_psnr,
            }, ckpt_path)
            print(f"  -> New best (val_psnr {val_psnr:.4f}), saved to {ckpt_path}")

    print("\nTraining complete. Best val PSNR:", best_psnr)


if __name__ == "__main__":
    main()
"""
with open(os.path.join(BASE_DIR, "train.py"), "w") as f:
    f.write(train_py)
print("Wrote", os.path.join(BASE_DIR, "train.py"))

In [ ]:
!pip freeze | grep -iE "^(torch|numpy|pandas|scikit-image|lpips|pillow|matplotlib)==" > {os.path.join(BASE_DIR, "requirements.txt")}
!cat {os.path.join(BASE_DIR, "requirements.txt")}

## Sanity check — confirm all 6 required repo components exist
Copy `evaluate.py`, `train.py`, `requirements.txt`, `checkpoints/model_v2_best.pth`, and `final_submission_outputs/` from `BASE_DIR` into your GitHub repo, plus a `README.md` with setup instructions.

In [ ]:
required = {
    "evaluate.py (standalone eval script)": os.path.join(BASE_DIR, "evaluate.py"),
    "train.py (training script)": os.path.join(BASE_DIR, "train.py"),
    "requirements.txt": os.path.join(BASE_DIR, "requirements.txt"),
    "checkpoints/model_v2_best.pth (trained weights)": CHECKPOINT_PATH,
    "final_submission_outputs/ (restored test outputs)": OUTPUT_DIR,
}
for label, path in required.items():
    exists = os.path.exists(path)
    print(f"{'OK ' if exists else 'MISSING'} - {label}: {path}")